# HAKE-MER — RoBERTa +M3 / +M4

**Backbone:** RoBERTa-base · batch 16, LR 5e-5, 4 epochs, 3 seeds.

Three blocks below (NRC M3, SenticNet M3, SenticNet M4). Each block: **train cell** (live epoch logs) → **summary** → **zip download**.

In [1]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [2]:
import subprocess
from getpass import getpass
from pathlib import Path

REPO, WORKDIR = "khalef-khalil/marii", Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = "--epochs 4 --batch-size 16 --lr 5e-5 --early-stopping-patience 0"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

/content/marii
851661b


In [3]:
!pip install -q -r requirements-train.txt
!chmod +x run_m1_m2_m3_campaign.sh run_m1_m2_m3_m4_campaign.sh

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.5 MB/s eta 0:00:00


## Block 1 — +M1+M2+M3 (NRC)

In [4]:
!./run_m1_m2_m3_campaign.sh --lexicon nrc --backbone roberta-base {TRAIN_FLAGS}

config.json: 100% 481/481 [00:00<00:00, 5.04MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 339kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 20.2MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 37.4MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 32.1MB/s]
README.md: 100% 9.40k/9.40k [00:00<00:00, 28.4MB/s]

simplified/train-00000-of-00001.parquet: downloading bytes:  14% 379k/2.77M [00:00<00:01, 1.22MB/s]
simplified/train-00000-of-00001.parquet: downloading bytes:  96% 2.66M/2.77M [00:00<00:00, 8.01MB/s,   ???B/s  ]
simplified/train-00000-of-00001.parquet: downloading bytes: 100% 2.73M/2.73M [00:00<00:00, 4.86MB/s,  266kB/s  ]
simplified/train-00000-of-00001.parquet: reconstructing file: 100% 2.77M/2.77M [00:00<00:00, 4.93MB/s,  271kB/s  ]

simplified/validation-00000-of-00001.par(…): downloading bytes:  32% 112k/350k [00:00<00:00, 538kB/s]
simplified/validation-00000-of-00001.par(…): downloading bytes: 100% 346k/346k [00:00<00:00, 1.44MB/s, 34.4kB/s  ]
simplified/validat

In [5]:
import json
from pathlib import Path

path = Path("reference/artifacts/m1_m2_m3_nrc_roberta_base_campaign.json")
c = json.loads(path.read_text(encoding="utf-8"))
print(path.name, "protocol:", c.get("protocol", {}))
if "lexicon_gate_abs" in c:
    print("  |g| mean:", c["lexicon_gate_abs"]["mean"])
for k, b in c["test_aggregate"].items():
    print(f"  {k}: {b['mean']:.4f} ± {b['std']:.4f}")

m1_m2_m3_nrc_roberta_base_campaign.json protocol: {'epochs': 4, 'batch_size': 16, 'lr': 5e-05, 'max_phrases': 4, 'phrase_max_length': 32, 'early_stopping_patience': 0}
  |g| mean: 0.03711949164668719
  f1_micro: 0.5769 ± 0.0163
  f1_macro: 0.4926 ± 0.0446
  exact_match: 0.4486 ± 0.0167
  map: 0.5148 ± 0.0174


In [6]:
import zipfile
from google.colab import files
from pathlib import Path

slug = "roberta_base"
campaign = Path(f"reference/artifacts/m1_m2_m3_nrc_{slug}_campaign.json")
out_name = "roberta_m3_nrc_campaign.zip"
zip_path = Path(f"/content/{out_name}")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(campaign, campaign.name)
    for m in sorted(Path("runs").glob(f"{slug}_seed*_m1_m2_m3_nrc/metrics.json")):
        zf.write(m, f"{m.parent.name}/{m.name}")
print(f"Download {out_name} ({zip_path.stat().st_size / 1e3:.1f} KB)")
files.download(str(zip_path))

Download roberta_m3_nrc_campaign.zip (6.6 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Block 2 — +M1+M2+M3 (SenticNet)

In [7]:
!./run_m1_m2_m3_campaign.sh --lexicon senticnet --backbone roberta-base {TRAIN_FLAGS}

Loading weights: 100% 197/197 [00:00<00:00, 15957.16it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Epoch 1: loss=0.1161 val_f1_macro=0.4022 val_f1_micro=0.5508 val_exact_match=0.4152 val_map=0.5141
Epoch 2: loss=0.0837 val_f1_macro=0.4645 val_f1_micro=0.5630 val_exact_match=0.4287 val_map=0.5460
Epoch 3: loss=0.0713 val_f1_macro=0.5081 val_f1_micro=0.584

In [8]:
import json
from pathlib import Path

path = Path("reference/artifacts/m1_m2_m3_senticnet_roberta_base_campaign.json")
c = json.loads(path.read_text(encoding="utf-8"))
print(path.name, "protocol:", c.get("protocol", {}))
if "lexicon_gate_abs" in c:
    print("  |g| mean:", c["lexicon_gate_abs"]["mean"])
for k, b in c["test_aggregate"].items():
    print(f"  {k}: {b['mean']:.4f} ± {b['std']:.4f}")

m1_m2_m3_senticnet_roberta_base_campaign.json protocol: {'epochs': 4, 'batch_size': 16, 'lr': 5e-05, 'max_phrases': 4, 'phrase_max_length': 32, 'early_stopping_patience': 0}
  |g| mean: 0.028631797060370445
  f1_micro: 0.5886 ± 0.0020
  f1_macro: 0.5235 ± 0.0030
  exact_match: 0.4631 ± 0.0040
  map: 0.5322 ± 0.0028


In [9]:
import zipfile
from google.colab import files
from pathlib import Path

slug = "roberta_base"
campaign = Path(f"reference/artifacts/m1_m2_m3_senticnet_{slug}_campaign.json")
out_name = "roberta_m3_senticnet_campaign.zip"
zip_path = Path(f"/content/{out_name}")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(campaign, campaign.name)
    for m in sorted(Path("runs").glob(f"{slug}_seed*_m1_m2_m3_senticnet/metrics.json")):
        zf.write(m, f"{m.parent.name}/{m.name}")
print(f"Download {out_name} ({zip_path.stat().st_size / 1e3:.1f} KB)")
files.download(str(zip_path))

Download roberta_m3_senticnet_campaign.zip (6.8 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Block 3 — +M1+M2+M3+M4 (SenticNet)

In [10]:
!./run_m1_m2_m3_m4_campaign.sh --lexicon senticnet --backbone roberta-base {TRAIN_FLAGS}

Loading weights: 100% 197/197 [00:00<00:00, 15021.32it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Epoch 1: loss=0.1183 val_f1_macro=0.3957 val_f1_micro=0.5539 val_exact_match=0.4204 val_map=0.5004
Epoch 2: loss=0.0834 val_f1_macro=0.4503 val_f1_micro=0.5829 val_exact_match=0.4554 val_map=0.5413
Epoch 3: loss=0.0718 val_f1_macro=0.5212 val_f1_micro=0.589

In [11]:
import json
from pathlib import Path

path = Path("reference/artifacts/m1_m2_m3_senticnet_m4_roberta_base_campaign.json")
c = json.loads(path.read_text(encoding="utf-8"))
print(path.name, "protocol:", c.get("protocol", {}))
if "lexicon_gate_abs" in c:
    print("  |g| mean:", c["lexicon_gate_abs"]["mean"])
for k, b in c["test_aggregate"].items():
    print(f"  {k}: {b['mean']:.4f} ± {b['std']:.4f}")

m1_m2_m3_senticnet_m4_roberta_base_campaign.json protocol: {'epochs': 4, 'batch_size': 16, 'lr': 5e-05, 'max_phrases': 4, 'phrase_max_length': 32, 'early_stopping_patience': 0}
  |g| mean: 0.028159112359086674
  f1_micro: 0.5880 ± 0.0012
  f1_macro: 0.5209 ± 0.0051
  exact_match: 0.4627 ± 0.0016
  map: 0.5332 ± 0.0045


In [12]:
import zipfile
from google.colab import files
from pathlib import Path

slug = "roberta_base"
campaign = Path(f"reference/artifacts/m1_m2_m3_senticnet_m4_{slug}_campaign.json")
out_name = "roberta_m4_senticnet_campaign.zip"
zip_path = Path(f"/content/{out_name}")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(campaign, campaign.name)
    for m in sorted(Path("runs").glob(f"{slug}_seed*_m1_m2_m3_senticnet_m4/metrics.json")):
        zf.write(m, f"{m.parent.name}/{m.name}")
print(f"Download {out_name} ({zip_path.stat().st_size / 1e3:.1f} KB)")
files.download(str(zip_path))

Download roberta_m4_senticnet_campaign.zip (6.9 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Integrity

Save executed `.ipynb` → `reference/training_records/step_roberta_ablation/colab/` · integrate each zip · rebuild RoBERTa ablation table in ch. 4.